# <b><font color='teal'>Homework 20</font></b>

Объедините прогнозы, полученные с помощью моделей ARIMA, SARIMA и Prophet, чтобы повысить точность предсказаний.

In [1]:
# Импорт библиотек
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
import numpy as np

In [7]:
# Загрузите датасет в датафрейм df_Dingling
from pathlib import Path

candidates = [
    Path('data/PRSA_Data_Dingling_20130301-20170228.csv'),
    Path('../data/PRSA_Data_Dingling_20130301-20170228.csv'),
]


display(df_Dingling)

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling
1,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling
2,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling
3,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling
4,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35059,35060,2017,2,28,19,11.0,11.0,2.0,2.0,200.0,99.0,11.7,1008.9,-13.3,0.0,NNE,1.3,Dingling
35060,35061,2017,2,28,20,13.0,13.0,2.0,2.0,200.0,101.0,10.9,1009.0,-14.0,0.0,N,2.1,Dingling
35061,35062,2017,2,28,21,9.0,14.0,2.0,2.0,200.0,102.0,9.5,1009.4,-13.0,0.0,N,1.5,Dingling
35062,35063,2017,2,28,22,10.0,12.0,2.0,2.0,200.0,97.0,7.8,1009.6,-12.6,0.0,NW,1.4,Dingling


In [9]:
# В df_Dingling создайте новый столбец datetime,
# в который соедините значения из столбцов year, month, day, hour
# Для этого используйте функцию pd.to_datetime
df_Dingling['datetime'] = pd.to_datetime(df_Dingling[['year', 'month', 'day', 'hour']])
display(df_Dingling)

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,datetime
0,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling,2013-03-01 00:00:00
1,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling,2013-03-01 01:00:00
2,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling,2013-03-01 02:00:00
3,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling,2013-03-01 03:00:00
4,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling,2013-03-01 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35059,35060,2017,2,28,19,11.0,11.0,2.0,2.0,200.0,99.0,11.7,1008.9,-13.3,0.0,NNE,1.3,Dingling,2017-02-28 19:00:00
35060,35061,2017,2,28,20,13.0,13.0,2.0,2.0,200.0,101.0,10.9,1009.0,-14.0,0.0,N,2.1,Dingling,2017-02-28 20:00:00
35061,35062,2017,2,28,21,9.0,14.0,2.0,2.0,200.0,102.0,9.5,1009.4,-13.0,0.0,N,1.5,Dingling,2017-02-28 21:00:00
35062,35063,2017,2,28,22,10.0,12.0,2.0,2.0,200.0,97.0,7.8,1009.6,-12.6,0.0,NW,1.4,Dingling,2017-02-28 22:00:00


In [10]:
# Созданный столбец сделайте индексом датафрейма (df.set_index())
df_Dingling.set_index('datetime', inplace=True)
display(df_Dingling)

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
datetime,,,,,,,,,,,,,,,,,,
2013-03-01 00:00:00,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling
2013-03-01 01:00:00,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling
2013-03-01 02:00:00,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling
2013-03-01 03:00:00,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling
2013-03-01 04:00:00,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2017-02-28 19:00:00,35060,2017,2,28,19,11.0,11.0,2.0,2.0,200.0,99.0,11.7,1008.9,-13.3,0.0,NNE,1.3,Dingling
2017-02-28 20:00:00,35061,2017,2,28,20,13.0,13.0,2.0,2.0,200.0,101.0,10.9,1009.0,-14.0,0.0,N,2.1,Dingling
2017-02-28 21:00:00,35062,2017,2,28,21,9.0,14.0,2.0,2.0,200.0,102.0,9.5,1009.4,-13.0,0.0,N,1.5,Dingling


In [11]:
# Поле PM2.5 выделите в отдельный объект типа Series и назовите его series_pm25
series_pm25 = df_Dingling['PM2.5']
display(series_pm25)

datetime
2013-03-01 00:00:00     4.0
2013-03-01 01:00:00     7.0
2013-03-01 02:00:00     5.0
2013-03-01 03:00:00     6.0
2013-03-01 04:00:00     5.0
                       ... 
2017-02-28 19:00:00    11.0
2017-02-28 20:00:00    13.0
2017-02-28 21:00:00     9.0
2017-02-28 22:00:00    10.0
2017-02-28 23:00:00    13.0
Name: PM2.5, Length: 35064, dtype: float64

In [ ]:
# Изучите series_pm25 на наличие отсутствующих значений


In [ ]:
# Примите решение об удалении/заполнении отсутствующих значений и выполните это действие


In [ ]:
# Агрегируйте данные по месяцам в переменную monthly_data


#### <b><font color='teal'>Модель ARIMA</font></b>

In [ ]:
# Постройте модель ARIMA (назовите model_pm25_arima)
# p - компонент авторегрессии (кол-во предыдущих значений)
# d - интегральный компонент (кол-во раз, которое необходимо продифференцировать данные)
# q - компонент скользящего среднего (сколько прошлых членов ошибки используется для прогнозирования)


In [ ]:
# Натренируйте модель
# Результат запишите в переменную model_pm25_arima_fit


In [ ]:
# В переменную forecast_arima_pm25 спрогнозируйте значения pm25 на ближайший год


In [ ]:
# Нарисуйте график временного ряда + график прогнозов


#### <b><font color='teal'>Модель SARIMA</font></b>

In [ ]:
# Создайте модель SARIMA с параметрами:
# p = 1
# d = 1
# q = 1
# P = 1
# D = 1
# Q = 1
# s = 12
# Назовите переменную model_pm25_sarima


In [ ]:
# Натренируйте модель на данных и запишите результат в model_pm25_sarima_fit


In [ ]:
# В переменную forecast_pm25_sarima спрогнозируйте данные на 12 месяцев вперед


In [ ]:
# Постройте график данных + прогнозы


#### <b><font color='teal'>Модель Prophet</font></b>

In [ ]:
# Из monthly_data создайте датафрейм df_pm25 со столбцами 'ds' и 'y'


In [ ]:
# Создайте экземпляр Prophet. Назовите model_prophet


In [ ]:
# Обучите модель


In [ ]:
# Создайте фрейм future_pm25 данных для хранения прогнозов
# с периодом 12 и с частотой в месяц (ME)


In [ ]:
# Сделайте прогнозы в переменную forecast_prophet_pm25


In [ ]:
# Визуализируйте прогноз и нарисуйте графики компонентов


#### <b><font color='teal'>Объединение прогнозов моделей</font></b>

In [ ]:
# Подготовьте прогнозы модели Prophet к формату данных других моделей
# Создайте Series из столбца yhat (причем последних 12 значений - tail(12))
# Индексами созданного Series будет столбец ds датафрейма forecast_prophet_pm25


In [ ]:
# В переменную mean_forecast вычислите среднее арифметическое всех трех прогнозов (ARIMA, SARIMA, Prophet)


In [ ]:
# Постройте график данных + усредненный прогноз
